In [361]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import os
import shutil
from sqlalchemy import create_engine
import pyodbc

import warnings
warnings.filterwarnings("ignore", category = DeprecationWarning)
warnings.filterwarnings("ignore", category = UserWarning)
warnings.filterwarnings("ignore", category = FutureWarning)

pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 500)

In [362]:
cnxn = pyodbc.connect(driver = '{ODBC Driver 17 for SQL Server}',
                      server = 'MISCPrdAdhocDB',
                      database = 'PRIME',
                      trusted_connection = 'yes')

In [363]:
def engine_wrapper(server = 'MISCPrdAdhocDB',
                  database = 'PRIME'):
    engine = create_engine('mssql+pyodbc://' + server + '/' + database + '?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')
    return engine

engine = engine_wrapper()

In [364]:
# load table util
def insert_to_prime_table(engine, cnxn, table_name, df_to_load, with_pkid = True, batch_size = 25000):
    
    db_header = pd.read_sql_query(f"SELECT TOP 2* FROM {table_name}", engine)
    cols = ','.join(['[' + col + ']' for col in db_header.columns])
    if with_pkid == True:
        cols = ','.join(['[' + col + ']' for col in db_header.columns][1:])
        
    q_string = ','.join(['?']*len(cols.split(',')))
    insert_sql = f"INSERT INTO {table_name} ({cols}) VALUES ({q_string})"
    print(f'Total of {(len(df_to_load)//batch_size) + 1} chunks will get inserted.')
    
    cursor = cnxn.cursor()
    cursor.fast_executemany = True
    for i in range(0, len(df_to_load), batch_size):
        chunk_df = df_to_load[i: i+batch_size]
        cursor.executemany(insert_sql, chunk_df.values.tolist())
        cursor.commit()
        print(f'chunk [{i}, {i+batch_size-1}] insertion completed.')
    print('Insertion completed.')
    cursor.close()
    return 0

def truncate_table(cnxn, table_name):
    truncate_sql = f"TRUNCATE TABLE {table_name}"
    cursor = cnxn.cursor()
    cursor.execute(truncate_sql)
    cursor.commit()
    print('table truncated.')
    return 0

In [365]:
q_map = {'01': 'Q1', 
         '02': 'Q1', 
         '03': 'Q1',
         '04': 'Q2', 
         '05': 'Q2', 
         '06': 'Q2',
         '07': 'Q3', 
         '08': 'Q3', 
         '09': 'Q3',
         '10': 'Q4', 
         '11': 'Q4', 
         '12': 'Q4'}

def add_quarter(year_month):
    if pd.isnull(year_month):
        return np.nan
    month = year_month.split('-')[1]
    return q_map[month]

# move files

In [366]:
data_dir1 = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\temp export"
data_dir2 = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm"
data_dir3 = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\BaxterEmergency"
file_move_config = {'PO': {'name': 'DataTeamPRD_POline_DL.csv',
                           'destination': data_dir1},
                    'POR': {'name': 'PurchaseOrderReceiptLine_DL+-+Export+To.csv',
                           'destination': data_dir1},
                    'INVH': {'name': 'AP_invoices_backbone_DL+-+Export+To+CSV.csv',
                           'destination': data_dir1},
                    'INVCL': {'name': 'PRD_DataTeam_PayablesInvoicePayment_ZFCH.csv',
                           'destination': data_dir1},
                    'INVGL': {'name': 'GL+distribution+-+Export+To+CSV.csv',
                           'destination': data_dir1},
                    'ITRX': {'name': 'Inventory_Trx_and_Balance+-+Export+To+CS.csv',
                           'destination': data_dir1},
                    'ILOC': {'name': 'Inventory_Location.csv',
                           'destination': data_dir1},
                    'POREL': {'name': 'PRD_DataTeam_PurchaseOrder_RevisionRel.csv',
                           'destination': data_dir1},
                    'REQ2':{'name': 'ReqLine.csv',
                           'destination': data_dir1},
                    'CONT': {'name': 'ContractLineSimple_full+-+Export+To+CSV.csv',
                           'destination': data_dir1},
                    'CONTTST': {'name': 'ContractLineImport_TST.csv',
                           'destination': data_dir1},
                    'IUOM': {'name': 'ItemUOM.csv',
                           'destination': data_dir2},
                    'ITEM': {'name': 'Item.csv',
                           'destination': data_dir2},
                    'EDISUB': {'name': 'EDI+sub.csv',
                           'destination': data_dir2},
                    'MF': {'name': 'Manufacturers.csv',
                           'destination': data_dir2},
                    'SUPPLIER': {'name': 'Suppliers.csv',
                           'destination': data_dir2},
                    'VITEMS': {'name': 'VendorItems.csv',
                           'destination': data_dir2},
                    'PARITEM': {'name': 'Par+Items+DL.csv',
                           'destination': data_dir2},
                    'BAXTERAUD': {'name': 'ItemAudit_BaxterItem_All.csv',
                           'destination': data_dir3},
                    'FD5': {'name': 'FD5.csv',
                            'destination': data_dir2},
                    'CLERROR': {'name': 'ContractLineError.csv',
                                'destination': data_dir1},
                    'VLP': {'name': 'VendorLocation_Preprocessor.csv',
                            'destination': data_dir2},
                    'IRF': {'name': 'ItemReplenishFrom.csv',
                            'destination': data_dir2},
                    'GTIN': {'name': 'ItemGTIN.csv',
                             'destination': data_dir2},
                    'RLOC': {'name': 'requesting+location.csv',
                             'destination': data_dir2}}

In [367]:
download_path = r"C:\Users\dli2\Downloads"
for k, v in file_move_config.items():
    if os.path.exists(os.path.join(download_path, v['name'])):
        shutil.move(os.path.join(download_path, v['name']), os.path.join(v['destination'], v['name']))
        print(f'file {k} moved.')
    else:
        print(v['name'], 'file not found.')

file PO moved.
file POR moved.
file INVH moved.
file INVCL moved.
file INVGL moved.
file ITRX moved.
file ILOC moved.
file POREL moved.
file REQ2 moved.
ContractLineSimple_full+-+Export+To+CSV.csv file not found.
file CONTTST moved.
file IUOM moved.
file ITEM moved.
file EDISUB moved.
file MF moved.
file SUPPLIER moved.
file VITEMS moved.
file PARITEM moved.
file BAXTERAUD moved.
file FD5 moved.
file CLERROR moved.
file VLP moved.
file IRF moved.
file GTIN moved.
file RLOC moved.


# read in flat files

In [368]:
data_dir = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\temp export"
po_line = "DataTeamPRD_POline_DL.csv"
po_receipt = "PurchaseOrderReceiptLine_DL+-+Export+To.csv"
invoice_header = "AP_invoices_backbone_DL+-+Export+To+CSV.csv"
invoice_cash_ledger = "PRD_DataTeam_PayablesInvoicePayment_ZFCH.csv"
invoice_gl_distribution = "GL+distribution+-+Export+To+CSV.csv"
inventory_location = "Inventory_Location.csv"
inventory_trx = "Inventory_Trx_and_Balance+-+Export+To+CS.csv"
requisition_line = "ReqLine.csv"
po_rev = r"PRD_DataTeam_PurchaseOrder_RevisionRel.csv"

contract_line = "ContractLineSimple_full+-+Export+To+CSV.csv"
contract_line_error = "ContractLineError.csv"
# contract_line = "ContractLineSimple.csv"
# contract_line_import = "ContractLineImport_TST.csv"

In [369]:
def type_conversion(target_type, df, columns):
    if target_type == 'date':
        for col in columns:
            converted_d = pd.to_datetime(df[col], errors = 'coerce')
            df[col] = converted_d.fillna(pd.to_datetime('1900-01-01'))
            df[col] = pd.to_datetime(df[col])
            df[col] = df[col].dt.date
    elif target_type == 'datetime':
        for col in columns:
            converted_dt = pd.to_datetime(df[col], errors = 'coerce')
            df[col] = converted_dt.fillna(pd.to_datetime('1900-01-01'))
    elif target_type == 'float':
        for col in columns:
            df[col] = df[col].apply(lambda x: float(str(x).strip().replace(',',''))
                                                            if not pd.isnull(x) else np.nan)
    elif target_type == 'int':
        for col in columns:
            df[col] = df[col].apply(lambda x: int(float(str(x).strip().replace(',','')))
                                                            if not pd.isnull(x) else None)
    return df

# inventory location

In [370]:
df = pd.read_csv(os.path.join(data_dir, inventory_location), dtype = str)
file_config = {'rename_cols': None,
               'drop_cols': ['Item.ReplacementItem', 'OrderMultiple'],
               'type_changes':{
               'date': [],
               'datetime': ['create stamp', 
                            'update stamp'],
               'float': ['Item.DefaultInventoryTransactionUOMMultiplier',
                         'Item.DefaultBuyUOMMultiplier',
                         'DefaultUnitCostInStockUOM',
                         'DerivedAverageCost'],
               'int': ['LeadtimeDays',
                       'LastLeadtime',
                       'StockOnHandQuantity',
                       'AvailableQuantity',
                       'AllocatedQuantity',
                       'DemandQuantity',
                       'OnOrderQuantity',
                       'ReorderPoint',
                       'MaximumOrderQuantity',
                       'MinimumOrderQuantity']},
                'rearrange_cols': None,
                'pk_check': ['Company', 'InventoryLocation', 'Item'],
                'staging_table_target': '[DM_MONTYNT\\dli2].inventory_location_stg'
               }

In [371]:
df = df.drop(columns = file_config['drop_cols'])

In [372]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [373]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-03 01:01:21 2023-09-10 23:00:00


In [374]:
# special transformation for inventory location
df.loc[:, 'report stamp'] = df['update stamp'].max()

In [375]:
# fillna with None or ''
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [376]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  InventoryLocation  Item  
3000     PWLRPICU           103943    1
                            104495    1
         PWLRPLD            101023    1
                            101050    1
                            101766    1
                                     ..
2010     IMNRSTRM           101820    1
                            101822    1
                            101823    1
                            101824    1
                            100316    1
Length: 72577, dtype: int64

In [377]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 2 chunks will get inserted.
chunk [0, 49999] insertion completed.
chunk [50000, 99999] insertion completed.
Insertion completed.


0

# Inventory Transaction

In [378]:
df = pd.read_csv(os.path.join(data_dir, inventory_trx), dtype = str)
file_config = {'rename_cols': None,
               'drop_cols': None,
               'type_changes':{
              'date': [],
              'datetime': ['create stamp', 
                           'update stamp',
                           'ActualTime'],
              'float': ['InventoryDistributionAmount',
                       'BaseCost',
                       'CurrentCost',
                       'TransientUnitCostValueInStockUOM',
                       'TransactionUOMMultiplier'],
              'int': ['StockOnHandQuantity',
                     'DerivedQuantityInStockUOM',
                     'DerivedQuantity',
                     'DerivedBeforeQuantity',
                     'DerivedAfterQuantityInStock',
                     'ItemLocation.OrderMultiple']},
               'rearrange_cols': ['TransactionSystemCode', 
                                  'InventoryTransaction.InventoryDocumentType',
                                  'ActualTime', 
                                  'Company', 
                                  'InventoryLocation', 
                                  'Bin', 
                                  'MultipleBins',
                                  'FromToCompanyLocationBin.FromToCompany',
                                  'FromToCompanyLocationBin.FromToLocation',
                                  'FromToCompanyLocationBin.FromToBin',
                                  'FromToCompanyLocationBin.RequestingLocation',
                                  'OffsetIndex', 
                                  'OffsetGL', 
                                  'InventoryIndex', 
                                  'InventoryGL',
                                  'InventoryTransaction', 
                                  'InventoryTransactionLine.LineNumber',
                                  'OriginatingTransactionDocument', 
                                  'OriginatingTransactionLine', 
                                  'Item',
                                  'Item.Description', 
                                  'InventoryDistributionAmount',
                                  'StockOnHandQuantity', 
                                  'BaseCost', 
                                  'CurrentCost',
                                  'TransientUnitCostValueInStockUOM', 
                                  'DerivedQuantityInStockUOM',
                                  'StockUOM', 
                                  'DerivedQuantity', 
                                  'TransactionUOM',
                                  'TransactionUOMMultiplier', 
                                  'ItemLocation.OrderMultiple',
                                  'DerivedBeforeQuantity',
                                  'DerivedAfterQuantityInStock', 
                                  'ToUOM', 
                                  'KitType', 
                                  'Status',
                                  'CreatedBy', 
                                  'LastUpdateBy', 
                                  'create stamp', 
                                  'update stamp'],
               'pk_check': ['Company', 'InventoryLocation', 
                            'InventoryTransaction', 'InventoryTransactionLine.LineNumber',
                            'Item', 'TransactionSystemCode'],
               'staging_table_target': '[DM_MONTYNT\\dli2].inventory_transaction_stg'
               }

In [379]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [380]:
# special transaformation for inventory transaction
# 2024-04-25 update, added offset account to data, index and GL need to get extracted from there
# df.loc[:, 'DistributionIndex'] = df['InventoryTransaction.DefaultDistributionAccount'].apply(lambda x: x.split('|')[1])
# df.loc[:, 'DistributionGL'] = df['InventoryTransaction.DefaultDistributionAccount'].apply(lambda x: x.split('|')[2])
df.loc[:, 'OffsetIndex'] = df['OffsetAccount'].apply(lambda x: x.split('|')[1])
df.loc[:, 'OffsetGL'] = df['OffsetAccount'].apply(lambda x: x.split('|')[2])
df.loc[:, 'InventoryIndex'] = df['InventoryAccount'].apply(lambda x: x.split('|')[1])
df.loc[:, 'InventoryGL'] = df['InventoryAccount'].apply(lambda x: x.split('|')[2])

In [381]:
df = df[file_config['rearrange_cols']].copy()

In [382]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-03 00:01:07 2026-05-29 00:03:02


In [383]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [384]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  InventoryLocation  InventoryTransaction  InventoryTransactionLine.LineNumber  Item    TransactionSystemCode
3000     IWLROR             556725                3                                    104671  Inventory Control        1
                                                  4                                    101113  Inventory Control        1
                                                  6                                    106460  Inventory Control        1
                                                  7                                    105298  Inventory Control        1
                                                  8                                    105312  Inventory Control        1
                                                                                                                       ..
         IWLRSTRM           555831                1                                    102921  Inventory Control        1
                             

In [385]:
df[(df['OffsetIndex'] == '') & (df['InventoryTransaction.InventoryDocumentType'] == 'Inventory Issue')]

,TransactionSystemCode,InventoryTransaction.InventoryDocumentType,ActualTime,Company,InventoryLocation,Bin,MultipleBins,FromToCompanyLocationBin.FromToCompany,FromToCompanyLocationBin.FromToLocation,FromToCompanyLocationBin.FromToBin,FromToCompanyLocationBin.RequestingLocation,OffsetIndex,OffsetGL,InventoryIndex,InventoryGL,InventoryTransaction,InventoryTransactionLine.LineNumber,OriginatingTransactionDocument,OriginatingTransactionLine,Item,Item.Description,InventoryDistributionAmount,StockOnHandQuantity,BaseCost,CurrentCost,TransientUnitCostValueInStockUOM,DerivedQuantityInStockUOM,StockUOM,DerivedQuantity,TransactionUOM,TransactionUOMMultiplier,ItemLocation.OrderMultiple,DerivedBeforeQuantity,DerivedAfterQuantityInStock,ToUOM,KitType,Status,CreatedBy,LastUpdateBy,create stamp,update stamp


In [386]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# POReceipt

In [387]:
df = pd.read_csv(os.path.join(data_dir, po_receipt), dtype = str)

In [388]:
file_config = {'rename_cols': None,
               'drop_cols': ['PurchaseOrder.update stamp'],
               'type_changes':{
               'date': ['PurchaseOrder.MMAHSPOReleaseDate', 
                        'ReceivedDate'],
               'datetime': ['create stamp', 
                            'update stamp'],
               'float': ['OriginalReceivedQuantity', 'OriginalUnitCost',
                        'EnteredReceivedQuantity', 'MatchUnitCost'],
               'int': ['(ReceivedDate - PurchaseOrder.MMAHSPOReleaseDate)']},
               'rearrange_cols': None,
               'pk_check': ['Company', 'PurchaseOrder', 
                            'PurchaseOrderLine.LineNumber', 'PurchaseOrderReceipt'],
               'staging_table_target': '[DM_MONTYNT\\dli2].purchaseorderreceipt_line_stg'
               }

In [389]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [390]:
df = df.drop(columns = file_config['drop_cols'])

In [391]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-02 23:48:00 2026-05-27 00:00:08


In [392]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [393]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  PurchaseOrder  PurchaseOrderLine.LineNumber  PurchaseOrderReceipt
3000     3000513458     5                             419867                  1
         3000513459     1                             419868                  1
                        2                             419868                  1
                        3                             419868                  1
                        4                             419868                  1
                                                                             ..
2010     2010126440     2                             29261                   1
                        4                             29261                   1
3000     3000513460     3                             419869                  1
                        4                             419869                  1
2000     2000102373     1                             1579                    1
Length: 24163, dtype: int64

In [394]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Invoice Header

In [395]:
df = pd.read_csv(os.path.join(data_dir, invoice_header), dtype = str)

In [396]:
file_config = {'rename_cols': None,
               'drop_cols': None,
               'type_changes':{
               'date': ['InvoiceDate', 
                        'DueDate', 
                        'CreateDate', 
                        'ReceiptOfInvoiceDate', 
                        'DistributionDate', 
                        'MatchDate', 
                        'TransientPaymentDate'], 
               'datetime': ['create stamp', 
                            'update stamp'],
               'float': ['InvoiceAmount.CurrencyAmount'],
               'int': []},
               'rearrange_cols': None,
               'pk_check': ['Company', 
                            'PayablesInvoice'],
               'staging_table_target': '[DM_MONTYNT\\dli2].payablesinvoice_header_stg'}

In [397]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [398]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-02 19:26:27 2026-05-27 02:45:08


In [399]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [400]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  PayablesInvoice
4200     3268               1
         3271               1
         3272               1
         3273               1
         3274               1
                           ..
2000     30178              1
         3138               1
         33455              1
         33456              1
1000     103                1
Length: 83585, dtype: int64

In [401]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 2 chunks will get inserted.
chunk [0, 49999] insertion completed.
chunk [50000, 99999] insertion completed.
Insertion completed.


0

# Invoice Payment

In [402]:
df = pd.read_csv(os.path.join(data_dir, invoice_cash_ledger), dtype = str)

In [403]:
file_config = {'rename_cols': None,
               'drop_cols': None,
               'type_changes':{
               'date': ['PayablesInvoice.TransientPaymentDate', 
                        'VoidDate',
                        'CheckDate', 
                        'DerivedInvoiceDate'], 
               'datetime': ['create stamp', 
                            'update stamp',
                            'PayablesInvoice.update stamp'],
               'float': ['BankCheckAmount'],
               'int': []},
               'rearrange_cols': ['Company', 
                                  'DerivedCompanyRepresentativeText',
                                  'PayablesInvoice', 
                                  'TransactionNumber',
                                  'PayablesInvoice.TransientPaymentNumber',
                                  'PayablesInvoice.TransientPaymentDate',
                                  'CheckDate', 
                                  'VoidDate',
                                  'BankCheckAmount',
                                  'PaymentStatus', 
                                  'BankStatus',
                                  'DerivedInvoice',
                                  'PayablesInvoice.InvoiceStatus',
                                  'DerivedInvoiceDate', 
                                  'DerivedInvoiceType',
                                  'Vendor', 
                                  'CashCode', 
                                  'BankTransactionCode',
                                  'IsApplied',
                                  'CreatedBy', 
                                  'LastUpdateBy',
                                  'create stamp', 
                                  'update stamp'],
               'pk_check': ['Company', 
                            'PayablesInvoice', 
                            'PayablesInvoice.TransientPaymentNumber', 
                            'VoidDate',
                            'create stamp'],
               'staging_table_target': '[DM_MONTYNT\\dli2].payablesinvoice_payment_stg'}

In [404]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [405]:
print(df['PayablesInvoice.update stamp'].max(), df['PayablesInvoice.update stamp'].min())

2026-06-02 19:26:27 2026-05-27 02:45:08


In [406]:
df = df[file_config['rearrange_cols']].copy()

In [407]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-02 23:07:15 2023-10-16 09:43:43


In [408]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [409]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  PayablesInvoice  PayablesInvoice.TransientPaymentNumber  VoidDate    create stamp       
4200     3277             4200200546                              1900-01-01  2026-05-11 11:18:38    1
2000     26745                                                    1900-01-01  2025-03-04 17:35:38    1
         26746                                                    1900-01-01  2025-03-04 17:35:47    1
         26747                                                    1900-01-01  2025-03-04 17:35:57    1
         26748                                                    1900-01-01  2025-03-04 17:36:08    1
                                                                                                    ..
4200     3272             4200200529                              1900-01-01  2026-04-20 10:08:18    1
         3273             4200200530                              1900-01-01  2026-04-28 11:27:24    1
         3274             4200200547                              1900-01-01  

In [410]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 2 chunks will get inserted.
chunk [0, 49999] insertion completed.
chunk [50000, 99999] insertion completed.
Insertion completed.


0

# Invoice GL Distribution

In [411]:
# data_dir = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\temp archive"
# invoice_gl_distribution = "GL+distribution+-+Export+To+CSV.csv"
df = pd.read_csv(os.path.join(data_dir, invoice_gl_distribution), dtype = str)

In [412]:
file_config = {'rename_cols': None,
               'drop_cols': None,
               'type_changes':{
               'date': ['DistributionDate', 
                        'PayablesInvoice.InvoiceDate', 
                        'PayablesInvoice.ReceiptOfInvoiceDate'], 
               'datetime': ['create stamp', 
                            'update stamp',
                            'PayablesInvoice.update stamp'],
               'float': ['DerivedFundDistributionOpenAmount', 
                         'DerivedGLTransactionAmount', 
                         'DerivedQuantity',
                         'DerivedUnitCost'],
               'int': []},
               'rearrange_cols': ['Company', 
                                  'PayablesInvoice', 
                                  'PayablesInvoiceDistribution',
                                  'AccountingEntity', 
                                  'AdjustedDistribution', 
                                  'AssetFlag',
                                  'DerivedFundDistributionOpenAmount', 
                                  'DerivedGLTransactionAmount',
                                  'DistributionDate', 
                                  'CreatedBy', 
                                  'Description',
                                  'DistributionType',
                                  'DistributionAccount.FinanceDimension1', 
                                  'DistributionAccount.FinanceDimension1.Description',
                                  'DistributionAccount.FinanceDimension3', 
                                  'DistributionAccount.FinanceDimension3.Description',
                                  'DistributionAccount.FinanceDimension5', 
                                  'DistributionAccount.FinanceDimension5.Description',
                                  'DistributionAccount.GeneralLedgerChartAccount.Account', 
                                  'DistributionAccount.GeneralLedgerChartAccount.AccountDescription',
                                  'DistributionAccount', 
                                  'GLJournalizeGroup', 
                                  'Invoice', 
                                  'PayablesInvoice.InvoiceDate', 
                                  'DerivedStatus', 
                                  'DerivedInvoiceType',
                                  'Vendor', 
                                  'PayablesInvoice.DerivedVendorName',
                                  'PayablesInvoice.MatchInvoice',
                                  'DerivedPlainTextComment',
                                  'PurchaseOrder', 
                                  'PurchaseOrderLine',
                                  'PurchaseOrderLine.MMAHSGeneralLedgerCategoryInventoryAccountGeneralLedgerChartAccount',
                                  'PurchaseOrderLine.MMAHSGeneralLedgerCategoryGeneralLedgerChartAccountDescription',
                                  'DerivedQuantity', 
                                  'DerivedUOM', 
                                  'DerivedUnitCost',
                                  'DerivedItemDescription',
                                  'PurchasingContract', 
                                  'ContractLine',
                                  'create stamp', 
                                  'update stamp'],
               'pk_check': ['Company', 
                            'PayablesInvoice', 
                            'PayablesInvoiceDistribution'],
               'staging_table_target': '[DM_MONTYNT\\dli2].payablesinvoice_distribution_stg'}

In [413]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [414]:
print(df['PayablesInvoice.update stamp'].max(), df['PayablesInvoice.update stamp'].min())

2026-06-02 19:26:27 2026-05-29 00:00:37


In [415]:
df = df[file_config['rearrange_cols']].copy()

In [416]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-02 17:50:17 2023-10-01 13:05:27


In [417]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [418]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  PayablesInvoice  PayablesInvoiceDistribution
4200     3272             8                              1
                          9                              1
         3273             1                              1
                          2                              1
                          3                              1
                                                        ..
         3277             6                              1
                          7                              1
                          8                              1
                          9                              1
1000     103              1                              1
Length: 514589, dtype: int64

In [419]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 11 chunks will get inserted.
chunk [0, 49999] insertion completed.
chunk [50000, 99999] insertion completed.
chunk [100000, 149999] insertion completed.
chunk [150000, 199999] insertion completed.
chunk [200000, 249999] insertion completed.
chunk [250000, 299999] insertion completed.
chunk [300000, 349999] insertion completed.
chunk [350000, 399999] insertion completed.
chunk [400000, 449999] insertion completed.
chunk [450000, 499999] insertion completed.
chunk [500000, 549999] insertion completed.
Insertion completed.


0

# PO Line

In [420]:
df = pd.read_csv(os.path.join(data_dir, po_line), dtype = str)

In [421]:
file_config = {'rename_cols': {'MMAHSPurchaseOrderLineDistributionFinanceDimension1': 'FD1',
                               'MMAHSPurchaseOrderLineDistributionFinanceDimension1Description': 'FD1Text',
                               'MMAHSPurchaseOrderLineDistributionFinanceDimension3': 'FD3',
                               'MMAHSPurchaseOrderLineDistributionFinanceDimension4': 'FD4',
                               'MMAHSPurchaseOrderLineDistributionFinanceDimension5': 'FD5',
                               'MMAHSPurchaseOrderLineDistributionProject': 'Project',
                               'MMAHSPurchaseOrderLineDistributionGeneralLdegerChartAccount': 'GL',
                               'MMAHSPurchaseOrderLineDistributionGeneralLedgerChartAccountDescription': 'GLText',
                               'MMAHSGeneralLedgerCategoryInventoryAccountFinanceDimension1': 'InventoryFD1',
                               'last PurchaseOrderLineSourcesRel.PurchaseOrderLineSource.SourceDocumentOrigin': 'SourceDocumentOrigin',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.PurchaseOrderLineSource.SourceDocumentNumeric': 'Requisition',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.PurchaseOrderLineSource.SourceDocumentLineNumber': 'RequisitionLine',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.RequisitionLineRel.ApprovedRejectedDate': 'RequisitionApprovedRejectedDate',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.RequisitionLineRel.Approved': 'RequisitionApproved',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.RequisitionLineRel.ApprovingRejectingOperatorID': 'RequisitionApprovingRejectingOperatorID',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.RequestingLocation': 'RequisitionRequestingLocation',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.RequisitionToPurchaseOrderDays': 'RequisitionToPurchaseOrderDays',
                               'last PurchaseOrderLineSourcesRel.RequestingLocation.RequisitionApprovalType': 'RequisitionApprovalType',
                               'last LineSourcesFromOrderEntryAndRequisitionRel.RequisitionRel.NeedsApproval': 'RequisitionNeedsApproval' 
                              },
               'drop_cols': None,
               'type_changes':{
               'date': ['MMAHSPurchaseOrderDate', 
                        'PurchaseOrder.MMAHSPOReleaseDate', 
                        'EarlyDeliveryDate',
                        'Contract.EffectiveDate', 
                        'Contract.ExpirationDate',
                        'RequisitionApprovedRejectedDate'], 
               'datetime': ['create stamp', 
                            'update stamp',
                            'PurchaseOrder.update stamp'],
               'float': ['PurchaseOrder.TotalOrderAmount', 
                         'CalculateExtendedAmount', 
                         'DerivedOpenToReceiveTotalAmount',
                         'DerivedReceivedNotInvoicedTotalAmount', 
                         'DerivedOpenQty', 
                         'DerivedReceivedNotInvoicedQuantity',
                         'Quantity', 
                         'VendorBuyUnitCost', 
                         'EnteredBuyUOMMultiplier',
                         'ReceivedQuantity',
                         'CancelQuantity'],
               'int': []},
               'rearrange_cols': ['System', 
                                  'AP pay lv1', 
                                  'AP pay lv3', 
                                  'Company', 
                                  'Name',
                                  'PurchaseOrder', 
                                  'PurchaseOrder.Reference1', 
                                  'MMAHSPOCode', 
                                  'ServiceCode',
                                  'MMAHSPurchaseOrderDate', 
                                  'PurchaseOrder.DerivedPurchaseOrderStatus',
                                  'PurchaseOrder.IsReleased', 
                                  'PurchaseOrder.MMAHSPOReleaseDate',
                                  'Year', 
                                  'YearMonth', 
                                  'Quarter',
                                  'PurchaseOrder.BuyerName', 
                                  'PurchaseOrder.DerivedRequesterName',
                                  'PurchaseOrder.ShipToLocation', 
                                  'PurchaseOrder.ShipToLocationName',
                                  'Vendor', 
                                  'Vendor.VendorName',
                                  'PurchaseOrder.TotalOrderAmount',
                                  'LineNumber',
                                  'PurchaseOrderLine',
                                  'EarlyDeliveryDate', 
                                  'PurchaseOrderLineLifeCycleState', 
                                  'LineReleased',
                                  'FD5', 
                                  'FD5Text', 
                                  'Project', 
                                  'InventoryFD1', 
                                  'TransientInventoryLocation',
                                  'FD1', 
                                  'FD1Text',
                                  'FD3', 
                                  'FD3Text',
                                  'GL', 
                                  'GLText',
                                  'FD4',
                                  'Item', 
                                  'ItemType',
                                  'VendorItem', 
                                  'Description',
                                  'Manufacturer', 
                                  'ManufacturerNumber',
                                  'EnteredBuyUOM', 
                                  'Quantity', 
                                  'VendorBuyUnitCost',
                                  'CalculateExtendedAmount',
                                  'Item.Manufacturer', 
                                  'Item.ManufacturerDescription', 
                                  'Item.DerivedStrippedManufacturerNumber', 
                                  'Item.StockUOM',
                                  'EnteredBuyUOMMultiplier', 
                                  'Item.OnContractDisplay',
                                  'Contract', 
                                  'ContractLine', 
                                  'Contract.WorkingContractID', 
                                  'Contract.ContractAndName', 
                                  'Contract.ContractClassification',
                                  'Contract.EffectiveDate', 
                                  'Contract.ExpirationDate',
                                  'IsOpenForReceivingIncludingUnreleased',
                                  'DerivedOpenToReceiveTotalAmount', 
                                  'DerivedOpenQty',
                                  'ReceivedNotInvoicedQuantityExists',
                                  'DerivedReceivedNotInvoicedTotalAmount', 
                                  'DerivedReceivedNotInvoicedQuantity', 
                                  'ReceivedQuantity',
                                  'CancelQuantity',
                                  'ItemIsBackordered',
                                  'CostCode',
                                  'CostOption',
                                  'PurchaseOrder.IssueMethod',
                                  'PurchaseOrder.PurchaseOrderLineCommentsRel.CommentText',
                                  'SourceDocumentOrigin',
                                  'Requisition',
                                  'RequisitionLine',
                                  'RequisitionApprovedRejectedDate',
                                  'RequisitionApproved',
                                  'RequisitionApprovingRejectingOperatorID',
                                  'RequisitionRequestingLocation',
                                  'RequisitionApprovalType',
                                  'EXC_FLAG',
                                  'create stamp', 
                                  'update stamp'],
               'pk_check': ['PurchaseOrder', 
                            'LineNumber'],
               'staging_table_target': '[DM_MONTYNT\\dli2].purchaseorder_line_stg'}

In [422]:
df = df.rename(columns = file_config['rename_cols'])

In [423]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [424]:
print(df['PurchaseOrder.update stamp'].max(), df['PurchaseOrder.update stamp'].min())

2026-06-02 22:10:52 2026-05-27 02:00:04


In [425]:
print(df['create stamp'].max(), df['create stamp'].min())

2026-06-02 20:16:17 2023-10-02 10:32:10


In [426]:
# special transformation for the file
map_dir = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\S13-APPO by vendor\APPO compilation"

# 1. company hierachy
company_map = \
pd.read_csv(os.path.join(map_dir, 'company_map.csv'), 
            dtype = str)
cmap = \
df[['Company']].drop_duplicates().merge(company_map[['Company', 'Name', 'AP pay lv3', 'AP pay lv1']].dropna().drop_duplicates(
    subset = ['Company']), on = ['Company'], how = 'left')

# if returns true, meaning no need to update the company map, everything is mapped
if all([i == 0 for i in cmap.isna().sum().values.tolist()]):
    df = df.merge(cmap, on = ['Company'], how = 'left')
else:
    print('check the company not mapped and fill them in.')
    display(cmap)

In [427]:
# move fd5 to folders to refresh data
source_file1 = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\FD5.csv"
des_path1 = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\S13-APPO by vendor\APPO compilation\FD5.csv"
des_path2 = r"I:\Procurement PMO\dli2\BaxterReport\mapping file for POLine\FD5.csv"

shutil.copy2(source_file1, des_path1)
print("file copied to path1")
shutil.copy2(source_file1, des_path2)
print("file copied to path2")

file copied to path1
file copied to path2


In [428]:
# move fd1/fd3 to folders to refresh data
source_file2 = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\FD1.csv"
des_path3 = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\S13-APPO by vendor\APPO compilation\FD1.csv"
des_path4 = r"I:\Procurement PMO\dli2\BaxterReport\mapping file for POLine\FD1.csv"

shutil.copy2(source_file2, des_path3)
print("file copied to path1")
shutil.copy2(source_file2, des_path4)
print("file copied to path2")

file copied to path1
file copied to path2


In [429]:
# 2. resolve financial dimensions that are unknown
fd1 = pd.read_csv(os.path.join(map_dir, "FD1.csv"), dtype = str)
fd2 = pd.read_csv(os.path.join(map_dir, "FD2.csv"), dtype = str)
fd3 = pd.read_csv(os.path.join(map_dir, "FD3.csv"), dtype = str)
fd4 = pd.read_csv(os.path.join(map_dir, "FD4.csv"), dtype = str)
fd5 = pd.read_csv(os.path.join(map_dir, "FD5.csv"), dtype = str)

# no desc all belong to group 2_C30005, 30050 - 2_C30049, if unkown, can check fd_map
# for cost center below, FD5 will simply inherit the FD1 Description
FD1_desc_fill = {'30005': 'Financial Services Executive Payroll',
                 '30440': 'Financial Services Executive Offices 4th Floor',
                 '30441': 'Financial Services Executive Payroll Oop2',
                 '30537': 'Financial Services Senior VP CMO Payroll',
                 '30601': 'Financial Services Executive Health',
                 '30050': 'Administrative Shared Services Executive',
                 '30069': 'Administrative_Shared Services Administration',
                 '30851': 'Financial Services Office of SSVP and Chief Legal Officer',}

# FD1
x1 = df[['FD1', 'FD1Text']].drop_duplicates().merge(fd1, 
                                        left_on = ['FD1'], 
                                        right_on = ['FinanceDimension1'], 
                                        how = 'left').sort_values(by = ['FinanceDimension1'])
display(x1[x1['FinanceDimension1'].isnull()])

# FD3
x3 = df[['FD3']].drop_duplicates().merge(fd3, 
                                        left_on = ['FD3'], 
                                        right_on = ['FinanceDimension3'], 
                                        how = 'left').sort_values(by = ['FinanceDimension3'])
display(x3[x3['FinanceDimension3'].isnull()])

# FD4
x4 = df[['FD4']].drop_duplicates().merge(fd4, 
                                        left_on = ['FD4'], 
                                        right_on = ['FinanceDimension4'], 
                                        how = 'left').sort_values(by = ['FinanceDimension4'])
display(x4[x4['FinanceDimension4'].isnull()])

# FD5
x5 = df[['FD5']].drop_duplicates().merge(fd5, 
                                        left_on = ['FD5'], 
                                        right_on = ['FinanceDimension5'], 
                                        how = 'left').sort_values(by = ['FinanceDimension5'])
display(x5[x5['FinanceDimension5'].isnull()])

,FD1,FD1Text,FinanceEnterpriseGroup,FinanceDimension1,DimensionType,DisplayDimension,Description,Active
10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405,30440,Financial Services Executive Offices 4th Floor,NaN,NaN,NaN,NaN,NaN,NaN
409,30005,Financial Services Executive Payroll,NaN,NaN,NaN,NaN,NaN,NaN


,FD3,FinanceEnterpriseGroup,FinanceDimension3,DimensionType,DisplayDimension,Description
3,NaN,NaN,NaN,NaN,NaN,NaN


,FD4,FinanceDimension4,FinanceEnterpriseGroup,DimensionType,DisplayDimension,Description
0,NaN,NaN,NaN,NaN,NaN,NaN


,FD5,FinanceEnterpriseGroup,FinanceDimension5,DimensionType,DisplayDimension,Description
10,NaN,NaN,NaN,NaN,NaN,NaN
571,30440998,NaN,NaN,NaN,NaN,NaN


In [430]:
# helper before filling
fd3_m = fd3[['FinanceDimension3', 'Description']].copy()
fd3_m.columns = ['FD3', 'FD3Text']
fd5_m = fd5[['FinanceDimension5', 'Description']].copy()
fd5_m.columns = ['FD5', 'FD5Text']

# merge to get FD desc
df = df.merge(fd3_m, on = ['FD3'], how = 'left').merge(fd5_m, on = ['FD5'], how = 'left')

# treat inventory items, and make FD5 empty take FD1 description
inventory_index = df[~df['InventoryFD1'].isnull()].index
df.loc[inventory_index, 'FD5'] = 'INVENTORY' + ' - ' + df.loc[inventory_index, 'TransientInventoryLocation']
df.loc[inventory_index, 'FD5Text'] = df.loc[inventory_index, 'FD5']
df.loc[:, 'FD5Text'] = df['FD5Text'].fillna(df['FD1Text'])

In [431]:
# 3. mark the legacy ERP transferred PO
def remove_leading_zeros(st):
    if pd.isnull(st) or st == 'nan':
        return ''
    i = 0
    while i < len(st):
        if st[i] == '0':
            i += 1
        else:
            break
    return st[i:]

df.loc[:, 'PurchaseOrder.Reference1'] = df['PurchaseOrder.Reference1'].\
apply(lambda x: remove_leading_zeros(str(x).strip()))

In [432]:
# helper column to mark the reference1 that has more than two letters and not equal to SV
# '' -- keep
# not valid legacy order number -- keep
# valid legacy order number -- drop

def reference_checker(st):
    if st in ['4502656845 & 4502389074', '4502702803 - 4502536259', '4502689964 & 4502179112']:
        return False
    if st == '':
        return True
        
    cnt = 0
    for i in st:
        if i.isdigit():
            pass
        else:
            cnt += 1
    if cnt >= 2 and not (st.startswith('SV') or st.startswith('SA') or st.startswith('RJ') or st.startswith('KH') \
                        or st.startswith('DSA') or st.startswith('EB') or st.startswith('ST') or st.startswith('OK') \
                        or st.startswith('LS') or st.startswith('PO')):
        return True
    return False

# false -- transferred
# true -- new PO on Infor
df.loc[:, 'Helper'] = df['PurchaseOrder.Reference1'].apply(lambda x: reference_checker(x))
#this list probably will grow for a while, when mark with this, the PO are not marked as transaferred from legacy
still_need_list = set(['2624102630', '2624101668', '2600104995', '2600104751',
                  '2600104626', '2600104586', '2600104585', '2600104250',
                  '2400108304', '2400108045', '2400107700', '2200101120',
                  '2600104585', '3000274905', '2400128210', '2400133727'])
still_need_po_ind = df[df['PurchaseOrder'].isin(still_need_list)].index
with_legacy_po_ind = df[df['PurchaseOrder.Reference1'] != ''].index

df.loc[still_need_po_ind, 'Helper'] = False

In [433]:
# use this to check and keep the still_need_list (non-transfer PO) list growing,
# if return something, check and add to above list when necessary, we expected to have empty df
df[(df['Helper'] == True) & (df['PurchaseOrder.Reference1'] != '')][['PurchaseOrder', 'PurchaseOrder.Reference1']].drop_duplicates().\
sort_values(by = ['PurchaseOrder.Reference1'])

,PurchaseOrder,PurchaseOrder.Reference1


In [434]:
# mark orders that are transferred from legacy system
legacy_ind = df[df['Helper'] == False].index
# mark orders that are not released
unreleased_ind = df[(df['PurchaseOrder.IsReleased'] == 'No') & 
                        (df['PurchaseOrder.DerivedPurchaseOrderStatus'] != 'Closed')].index
canceled_after_ind = df[(df['LineReleased'] == 'Yes') &  
                        (df['PurchaseOrder.DerivedPurchaseOrderStatus'] == 'Canceled')].index

df.loc[:, 'EXC_FLAG'] = 'Default'
df.loc[legacy_ind, 'EXC_FLAG'] = 'Remove - legacy PO transferred'
df.loc[unreleased_ind, 'EXC_FLAG'] = 'Remove - unreleased'
df.loc[canceled_after_ind, 'EXC_FLAG'] = 'Remove - released but canceled'

In [435]:
df.groupby(['EXC_FLAG']).agg({'CalculateExtendedAmount': 'sum',
                             'LineNumber':'count'})

,CalculateExtendedAmount,LineNumber
EXC_FLAG,,
Default,2.523339e+08,37477
Remove - legacy PO transferred,1.051407e+08,706
Remove - released but canceled,0.000000e+00,3
Remove - unreleased,1.096024e+07,173


In [436]:
# 4. system and reporting period fix
df.loc[:, 'System'] = 'INFOR'
df.loc[:, 'Year'] = df['PurchaseOrder.MMAHSPOReleaseDate'].apply(lambda x: str(x)[:4] if not pd.isnull(x) else np.nan)
df.loc[:, 'YearMonth'] = df['PurchaseOrder.MMAHSPOReleaseDate'].apply(lambda x: str(x)[:7] if not pd.isnull(x) else np.nan)
df.loc[:, 'Quarter'] = df['YearMonth'].apply(lambda x: add_quarter(x))

# make changes and adaptations to lines that are default but has no release date filled
year_month_approx_ind = df[(df['YearMonth'].isnull()) & (df['EXC_FLAG'] == 'Default')].index
df.loc[year_month_approx_ind, 'Year'] = df['MMAHSPurchaseOrderDate'].apply(lambda x: str(x)[:4])
df.loc[year_month_approx_ind, 'YearMonth'] = df['MMAHSPurchaseOrderDate'].apply(lambda x: str(x)[:7])
df.loc[year_month_approx_ind, 'Quarter'] = df['YearMonth'].apply(lambda x: add_quarter(x))

# check all default transactions are filled with YearMonth
df[(df['YearMonth'].isnull()) & (df['EXC_FLAG'] == 'Default')]

,Company,MMAHSPurchaseOrderDate,PurchaseOrder.IsReleased,PurchaseOrder.MMAHSPOReleaseDate,PurchaseOrder.DerivedPurchaseOrderStatus,PurchaseOrder,PurchaseOrder.Reference1,MMAHSPOCode,ServiceCode,Vendor,Vendor.VendorName,PurchaseOrder.TotalOrderAmount,LineNumber,PurchaseOrderLine,Item,VendorItem,Description,Manufacturer,ManufacturerNumber,ItemType,Item.Manufacturer,Item.ContextManufacturerNumber,Item.DerivedStrippedManufacturerNumber,Item.ManufacturerDescription,Item.StockUOM,Item.DefaultBuyUOMMultiplier,EnteredBuyUOMMultiplier,Item.OnContractDisplay,Contract,ContractLine,EnteredBuyUOM,Quantity,VendorBuyUnitCost,CalculateExtendedAmount,MatchedQuantity,DerivedNonServiceMatchedAmount,Contract.WorkingContractID,Contract.EffectiveDate,Contract.ExpirationDate,Contract.ContractAndName,Contract.ContractClassification,FD1,FD1Text,FD3,FD4,FD5,Project,GL,GLText,InventoryFD1,EarlyDeliveryDate,ShipToLocation,TransientInventoryLocation,PurchaseOrder.BuyerName,PurchaseOrder.DerivedRequesterName,PurchaseOrder.ShipToLocation,PurchaseOrder.ShipToLocationName,PurchaseOrderLineLifeCycleState,LineReleased,IsOpenForReceivingIncludingUnreleased,DerivedOpenToReceiveTotalAmount,DerivedOpenQty,ReceivedNotInvoicedQuantityExists,DerivedReceivedNotInvoicedQuantity,DerivedReceivedNotInvoicedTotalAmount,ItemIsBackordered,CancelQuantity,ReceivedQuantity,CostCode,CostOption,DerivedPurchaseOrderReceiptStatus,PurchaseOrder.PurchaseOrderLineCommentsRel.CommentText,PurchaseOrder.IssueMethod,SourceDocumentOrigin,Requisition,RequisitionLine,RequisitionApprovedRejectedDate,RequisitionApproved,RequisitionApprovingRejectingOperatorID,RequisitionRequestingLocation,RequisitionToPurchaseOrderDays,RequisitionApprovalType,PurchaseOrder.update stamp,create stamp,update stamp,Name,AP pay lv3,AP pay lv1,FD3Text,FD5Text,Helper,EXC_FLAG,System,Year,YearMonth,Quarter


In [437]:
df = df[file_config['rearrange_cols']].copy()

In [438]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-02 22:10:52 2023-11-16 14:39:26


In [439]:
print(df['create stamp'].max(), df['create stamp'].min())

2026-06-02 20:16:17 2023-10-02 10:32:10


In [440]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df[col] = df[col].fillna('')
        df[col] = df[col].apply(lambda x: str(x).strip())

In [441]:
df.groupby(file_config['pk_check']).size().sort_values()

PurchaseOrder  LineNumber
3000513482     4             1
               5             1
               6             1
               7             1
               8             1
                            ..
3010100470     1             1
3010100473     1             1
3010100474     1             1
3090100120     1             1
2000100152     3             1
Length: 38359, dtype: int64

In [442]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Requisition Line

In [443]:
df = pd.read_csv(os.path.join(data_dir, requisition_line), dtype = str)

In [444]:
file_config = {'rename_cols': {'last PurchaseOrderLineSourcesRel.PurchaseOrder': 'PO',
                               'last PurchaseOrderLineSourcesRel.PurchaseOrderLine': 'POLine',
                               'last PurchaseOrderLineSourcesRel.PurchaseOrderLine.PurchaseOrder.PurchaseOrderDate': 'PODate'},
               'drop_cols': ['Requisition.update stamp', 'ManufacturerNumber',
                             'PurchaseOrderLineSourcesRel.PurchaseOrder.update stamp'],
               'type_changes':{
               'date': ['ApprovedRejectedDate',
                        'PODate'], 
               'datetime': ['create stamp', 
                            'update stamp',
                            'Requisition.update stamp'],
               'float': ['ApprovalValue',
                        'DisplayQuantity',
                        'DerivedTotalStockQuantity',
                        'KilledQuantityInStockUOM',
                        'AcceptedQuantity',
                        'DerivedVoidedQuantity',
                        'UnitCost',
                        'TransactionUnitCost',
                        'LandedUnitCost',
                        'ContractLine.BaseCost'],
               'int': []},
               'rearrange_cols': None,
               'pk_check': ['Company', 
                            'Requisition', 
                            'RequisitionLine'],
               'staging_table_target': '[DM_MONTYNT\\dli2].requisition_line_stg'}

In [445]:
df = df.rename(columns = file_config['rename_cols'])

In [446]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [447]:
print(df['Requisition.update stamp'].max(), df['Requisition.update stamp'].min())

2026-06-03 05:30:33 2026-05-27 05:20:15


In [448]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-03 05:30:33 2023-10-27 09:18:40


In [449]:
df = df.drop(columns = file_config['drop_cols'])

In [450]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [451]:
# fix basecost UOM
df['ContractLine.BaseCostUOM'] = df['ContractLine.BaseCostUOM'].apply(lambda x: x.split(' ')[-1])

In [452]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  Requisition  RequisitionLine
3000     469477       1                  1
                      10                 1
                      11                 1
                      12                 1
                      13                 1
                                        ..
3010     484          1                  1
         485          1                  1
         486          1                  1
         487          1                  1
2000     1972         1                  1
Length: 70875, dtype: int64

In [453]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 2 chunks will get inserted.
chunk [0, 49999] insertion completed.
chunk [50000, 99999] insertion completed.
Insertion completed.


0

# ContractLine

In [535]:
df = pd.read_csv(os.path.join(data_dir, contract_line), dtype = str)

In [536]:
file_config = {'rename_cols': None,
               'drop_cols': None,
               'type_changes':{
               'date': ['EffectiveDate',
                        'ExpirationDate',
                        'Contract.EffectiveDate',
                        'Contract.ExpirationDate'], 
               'datetime': ['create stamp', 
                            'update stamp'],
               'float': ['BaseCost',
                         'DerivedUOMConversion',
                         'PerOrderMinimumQuantity'],
               'int': ['Priority']},
               'rearrange_cols': ['Contract.ContractStatus',
                                  'Contract.OnHold',
                                  'Contract.MMAHSOrganizationEID',
                                  'Contract.WorkingContractID',
                                  'Contract',
                                  'DerivedContractName',
                                  'Contract.ContractClassification',
                                  'Contract.ContractSource',
                                  'Supplier',
                                  'Vendor',
                                  'Vendor.VendorName',
                                  'ApPoPurchaseFrom.PurchaseFromLocation',
                                  'Contract.ManufacturerCodeDivision',
                                  'Manufacturer',
                                  'ContractLine',
                                  'OnHold',
                                  'ContractLineState',
                                  'ActiveLine',
                                  'ItemType', 
                                  'ItemNumber', 
                                  'VendorItem', 
                                  'DerivedStrippedVendorItem',
                                  'ManufacturerNumber',
                                  'DerivedStrippedManufacturerNumber',
                                  'ItemDescription', 
                                  'UOM', 
                                  'BaseCost', 
                                  'DerivedUOMConversion',
                                  'PerOrderMinimumQuantity',
                                  'Priority', 
                                  'ErrorsExist',
                                  'FromImport',
                                  'EffectiveDate', 
                                  'ExpirationDate',
                                  'Contract.EffectiveDate',
                                  'Contract.ExpirationDate', 
                                  'CommodityCode', 
                                  'UNSPSCCode',
                                  'PatientChargeable', 
                                  'ChargeNumber',
                                  'MMAHSPrimaryDI',
                                  'MMAHSCCDescription', 
                                  'GlobalTradeItemNumber',
                                  'DerivedFirstAccount',
                                  'FirstAccount',
                                  'create stamp.actor', 
                                  'update stamp.actor', 
                                  'create stamp',
                                  'update stamp'],
               'pk_check': ['Contract', 
                            'ContractLine'],
               'staging_table_target': '[DM_MONTYNT\\dli2].contractline_stg'}

In [537]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [538]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-03 03:36:06 2023-07-27 01:08:54


In [539]:
df.loc[:, 'FirstAccount'] = df['DerivedFirstAccount'].apply(lambda x: x.split('|')[3])

In [540]:
df = df[file_config['rearrange_cols']].copy()

In [541]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [542]:
df.groupby(file_config['pk_check']).size().sort_values()

Contract  ContractLine
999       992             1
          993             1
          994             1
          995             1
          996             1
                         ..
10        129             1
          13              1
          130             1
          131             1
          132             1
Length: 1482528, dtype: int64

In [543]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 100000)

table truncated.
Total of 15 chunks will get inserted.
chunk [0, 99999] insertion completed.
chunk [100000, 199999] insertion completed.
chunk [200000, 299999] insertion completed.
chunk [300000, 399999] insertion completed.
chunk [400000, 499999] insertion completed.
chunk [500000, 599999] insertion completed.
chunk [600000, 699999] insertion completed.
chunk [700000, 799999] insertion completed.
chunk [800000, 899999] insertion completed.
chunk [900000, 999999] insertion completed.
chunk [1000000, 1099999] insertion completed.
chunk [1100000, 1199999] insertion completed.
chunk [1200000, 1299999] insertion completed.
chunk [1300000, 1399999] insertion completed.
chunk [1400000, 1499999] insertion completed.
Insertion completed.


0

# Purchase Order Header

In [463]:
po_rev = r"PRD_DataTeam_PurchaseOrder_RevisionRel.csv"
df = pd.read_csv(os.path.join(data_dir, po_rev), dtype = str)

In [464]:
file_config = {'rename_cols': None,
               'drop_cols': None,
               'type_changes':{
               'date': ['MMAHSPOReleaseDate',
                        'PurchaseOrderDate',
                        'MMAHSPurchaseOrderFirstRevisionIssueDate'], 
               'datetime': ['create stamp', 
                            'update stamp'],
               'float': [],
               'int': ['PurchaseOrderRevision']},
               'rearrange_cols': None,
               'pk_check': ['PurchaseOrder'],
               'staging_table_target': '[DM_MONTYNT\\dli2].purchaseorder_revision_issuemethod_stg'}

# '[DM_MONTYNT\\dli2].purchaseorder_revision_issuemethod_stg'

In [465]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [466]:
print(df['update stamp'].max(), df['update stamp'].min())

2026-06-02 22:10:52 2026-05-27 02:00:04


In [467]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [468]:
df.groupby(file_config['pk_check']).size().sort_values()

PurchaseOrder
3010100152    1
3010100166    1
3010100172    1
3010100392    1
3010100458    1
             ..
2000103432    1
2000103433    1
2000103434    1
2000103471    1
2000100152    1
Length: 11169, dtype: int64

In [469]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 100000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 99999] insertion completed.
Insertion completed.


0

# To temp load some df

## For Baxter Item Emergency

In [470]:
tickdata = "ItemAudit_BaxterItem_All.csv"
baxter_dir = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\BaxterEmergency"

In [471]:
df = pd.read_csv(os.path.join(baxter_dir, tickdata), dtype = str)
file_config = {'rename_cols': None,
               'drop_cols': None,
               'type_changes':{
               'date': [],
               'datetime': ['UpdateDate'],
               'float': ['BeforeAllocatedQuantity',
                        'AfterAllocatedQuantity',
                        'BeforeIntransitQuantity',
                        'AfterIntransitQuantity',
                        'BeforeStockOnHandQuantity',
                        'AfterStockOnHandQuantity',
                        'BeforeInProcessQuantity',
                        'AfterInProcessQuantity',
                        'BeforeOnOrderQuantity',
                        'AfterOnOrderQuantity'],
               'int': []},
                'rearrange_cols': ['Company',
                                 'InventoryLocation',
                                 'Item',
                                 'ActionContext',
                                 'Actor',
                                 'UpdateDate',
                                 'ItemLocationAudit',
                                 'BeforeAllocatedQuantity',
                                 'AfterAllocatedQuantity',
                                 'BeforeIntransitQuantity',
                                 'AfterIntransitQuantity',
                                 'BeforeInProcessQuantity',
                                 'AfterInProcessQuantity',
                                 'BeforeStockOnHandQuantity',
                                 'AfterStockOnHandQuantity',
                                 'BeforeOnOrderQuantity',
                                 'AfterOnOrderQuantity'],
                'pk_check': ['Company', 'InventoryLocation', 'Item', 'ItemLocationAudit'],
                'staging_table_target': '[DM_MONTYNT\\dli2].baxter_item_audit_stg'
               }

In [472]:
for k, v in file_config['type_changes'].items():
    if k == 'date' and len(v) > 0:
        df = type_conversion('date', df, v)
    if k == 'datetime' and len(v) > 0:
        df = type_conversion('datetime', df, v)
    if k == 'float' and len(v) > 0:
        df = type_conversion('float', df, v)
    if k == 'int' and len(v) > 0:
        df = type_conversion('int', df, v)

In [473]:
print(df['UpdateDate'].max(), df['UpdateDate'].min())

2026-06-02 23:57:19 2026-05-24 08:32:36


In [474]:
df = df[file_config['rearrange_cols']].copy()

In [475]:
for col in df.columns:
    if col in file_config['type_changes']['date']:
        df[col] = df[col].fillna('0001-01-01')
    elif col in file_config['type_changes']['int']:
        df[col] = df[col].astype(object).where(df[col].notna(), None)
    elif col in file_config['type_changes']['float'] + file_config['type_changes']['datetime'] + ['report stamp']:
        continue
    else:
        df.loc[:, col] = df[col].fillna('')
        df.loc[:, col] = df[col].apply(lambda x: str(x).strip())

In [476]:
df.groupby(file_config['pk_check']).size().sort_values()

Company  InventoryLocation  Item    ItemLocationAudit
3000     IMOSSTRM           105759  118                  1
                                    119                  1
                                    120                  1
                            105760  46                   1
                                    47                   1
                                                        ..
                            105764  45                   1
                            105767  120                  1
                            107781  85                   1
         IPYCSTRM           100857  1                    1
2010     IMNRSTRM           101794  3026                 1
Length: 19271, dtype: int64

In [477]:
# upload
table_to_load = df
table_name = file_config['staging_table_target']
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Par Item

In [478]:
pari = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\Par+Items+DL.csv",
                  dtype = str)

In [479]:
for col in ['create stamp', 'update stamp']:
    pari[col] = pd.to_datetime(pari[col])

pari['BinSequence'] = pari['BinSequence'].astype(int)
pari['DefaultTransactionUOM.UOMConversion'] = pari['DefaultTransactionUOM.UOMConversion'].apply(lambda x: float(x.replace(',', '')))

In [480]:
# upload
table_to_load = pari
table_name = 'PLM.ParItemBin'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

## Vendor Item

In [481]:
amap = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\VendorItems.csv",
                     dtype = str)

In [482]:
amap.groupby(['Item', 'Vendor', 'VendorItem']).size().sort_values()

Item    Vendor   VendorItem  
114493  1040877  951             1
114494  1002925  5570            1
114495  1002925  5272            1
114496  1002925  5641            1
114497  1002925  8802            1
                                ..
114509  1001292  N301            1
114510  1000540  AMS-9294        1
114511  1037888  IME10011865     1
114512  1000931  AMB3611017CS    1
100000  1000931  EVSVEN          1
Length: 22637, dtype: int64

In [483]:
amap.loc[:, 'LastUpdate'] = datetime.today().strftime('%Y-%m-%d')

In [484]:
amap.columns

Index(['Item', 'VendorItemDescription', 'Vendor', 'Vendor.VendorName',
       'VendorItem', 'Manufacturer', 'ManufacturerNumber', 'VendorBuyUOM',
       'VendorBuyUOM.UOMConversion', 'Item.DefaultBuyUOM',
       'Item.DefaultBuyUOMMultiplier', 'Item.UNSPSCCode',
       'Item.UNSPSCCode.ItemGroup.Description', 'Item.CommodityCode',
       'Item.MMAHSCCDescription', 'Item.MajorPurchasingClass',
       'Item.MajorPurchasingClass.MajorClassDescription',
       'Item.MinorPurchasingClass', 'Item.MinorPurchasingClass.Description',
       'UseAsDefault', 'Active', 'Item.MajorPPEClass',
       'Item.MajorInventoryClass', 'Item.OnContractDisplay',
       'first ContractLineVenItemRel.Contract',
       'last ContractLineVenItemRel.Contract', 'create stamp', 'update stamp',
       'LastUpdate'],
      dtype='str')

In [485]:
float_cols = ['Item.DefaultBuyUOMMultiplier', 'VendorBuyUOM.UOMConversion']
for col in amap.columns:
    if col in float_cols:
        amap[col] = amap[col].apply(lambda x: float(x.replace(',','')) if not pd.isnull(x) else 0.0)
    else:
        continue
amap = amap.fillna('')        

In [486]:
datetime_cols = ['create stamp', 'update stamp']
for col in datetime_cols:
    amap[col] = pd.to_datetime(amap[col])

In [487]:
# upload
table_to_load = amap
table_name = '[DM_MONTYNT\\dli2].MDM_VENDORITEM'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Item UOM

In [488]:
itemUOM = pd.read_csv(r'C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\ItemUOM.csv',
                     dtype = str)

In [489]:
itemUOM.groupby(['Item', 'UnitOfMeasure']).size().sort_values()

Item    UnitOfMeasure
114485  EA               1
114486  EA               1
114487  EA               1
114488  EA               1
114489  EA               1
                        ..
114511  CA               1
        EA               1
114512  CA               1
        EA               1
100000  BX               1
Length: 25023, dtype: int64

In [490]:
float_cols = ['UOMConversion', 'PackingWeight', 'PackingVolume']
datetime_cols = ['create stamp', 'update stamp']
for col in itemUOM.columns:
    if col in float_cols:
        itemUOM[col] = itemUOM[col].apply(lambda x: float(str(x).replace(',', '')))
    elif col in datetime_cols:
        itemUOM[col] = pd.to_datetime(itemUOM[col])
    else:
        continue

itemUOM = itemUOM.fillna('')

In [491]:
# upload
table_to_load = itemUOM
table_name = '[DM_MONTYNT\\dli2].MDM_ITEMUOM'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Manufacturer

In [492]:
mfn = pd.read_csv(r'C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\Manufacturers.csv',
                  dtype = str)
mfn = mfn[['Manufacturer', 'MMAHSManufacturerEID', 'Description', 'Active']].copy()
mfn.loc[:, 'ReportDate'] = datetime.now().strftime('%Y-%m-%d')

In [493]:
mfn.groupby(['Manufacturer']).size().sort_values()

Manufacturer
3114    1
3115    1
3116    1
3117    1
3118    1
       ..
1039    1
1040    1
3123    1
3124    1
1000    1
Length: 1987, dtype: int64

In [494]:
# upload
table_to_load = mfn
table_name = '[DM_MONTYNT\\dli2].MDM_MANUFACTURER_NAME_INFOR'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Supplier

In [495]:
sup = pd.read_csv(r'C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\Suppliers.csv',
                  dtype = str)

In [496]:
sup.loc[:, 'Supplier'] = sup['RepresentativeText'].apply(lambda x: x.split(' - ')[0].strip())
sup = sup[['Supplier', 'RepresentativeText', 'SupplierName', 'Vendor', 'Vendor.VendorName', 'Vendor.VendorClass',
           'Active', 'HasBeenValidated']].copy()
sup = sup.fillna('')
sup.loc[:, 'ReportDate'] = datetime.now().strftime('%Y-%m-%d')

In [497]:
sup.groupby(['Supplier']).size().sort_values()

Supplier
974    1
975    1
976    1
977    1
978    1
      ..
988    1
99     1
998    1
999    1
1      1
Length: 1675, dtype: int64

In [498]:
# upload
table_to_load = sup
table_name = '[DM_MONTYNT\\dli2].MDM_SUPPLIER_NAME_INFOR'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# EDI SUB

In [499]:
edi_sub = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\EDI+sub.csv")
edi_sub.loc[:, 'ReportDate'] = datetime.now().strftime('%Y-%m-%d')
edi_sub.fillna('NA', inplace = True)

for col in edi_sub.columns:
    edi_sub.loc[:, col] = edi_sub[col].astype(str)

In [500]:
# upload
table_to_load = edi_sub
table_name = '[DM_MONTYNT\\dli2].MDM_EDI_SUB_UOM'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Item

In [501]:
item = pd.read_csv(r'C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\Item.csv',
                  dtype = str)

In [502]:
cols_to_take = ['Item', 'Active', 'ConsignCode', 'Consignment', 'CriticalItem', 
               'DefaultBuyUOM', 'DefaultBuyUOMMultiplier',
               'DefaultInventoryTransactionUOM', 'DefaultInventoryTransactionUOMMultiplier', 'StockUOM',
               'Description', 'Description3', 'Discontinued', 'GenericName', 'Implantable', 'Reusable', 'Sterile',
               'GTINForStockUOM', 'ItemGTINsRel.Active', 'HCPCSCode', 'ItemDescriptionAbbreviation', 'CommodityCode',
               'CommodityCode.CcDescription', 'MMAHSPrimaryDI',
               'MajorInventoryClass', 'MajorPPEClass', 'MajorPurchasingClass', 'MajorPurchasingClass.Description',
               'MinorInventoryClass', 'MinorPPEClass', 'MinorPurchasingClass',
               'Manufacturer', 'ManufacturerDescription', 'ManufacturerNumber']
item = item[cols_to_take].copy()

In [503]:
item.loc[:, 'ReportDate'] = datetime.now().strftime('%Y-%m-%d')
for col in ['DefaultBuyUOMMultiplier', 'DefaultInventoryTransactionUOMMultiplier']:
    item[col] = item[col].apply(lambda x: int(float(x.replace(',',''))))
item = item.fillna('')
item.groupby(['Item']).size().sort_values()

Item
114495    1
114496    1
114497    1
114498    1
114499    1
         ..
114509    1
114510    1
114511    1
114512    1
100000    1
Length: 13162, dtype: int64

In [504]:
# upload
table_to_load = item
table_name = '[DM_MONTYNT\\dli2].MDM_ITEM'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# CCX Header

In [506]:
import pathlib
download_path = pathlib.Path(r"C:\Users\dli2\Downloads")
dest_dir = pathlib.Path(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\ccx")
ccx_sync = f"searchSyncContract*.csv"

found_files = list(download_path.glob("searchSyncContract*.csv"))

file = found_files[-1]
print(file)

if os.path.join(download_path, file):
    shutil.move(str(file), os.path.join(str(dest_dir), file.name))
    print('moved.')

C:\Users\dli2\Downloads\searchSyncContract_3_6_2026_10_07_55 am.csv
moved.


In [507]:
ccx_sync = "searchSyncContract"
ccx_sync_file_dir = r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\ccx"
today = datetime.today().strftime('%Y-%m-%d %H:%M:%S')
sync_contract = None
for file in os.listdir(ccx_sync_file_dir):
    if ccx_sync in file:
        sync_contract = pd.read_csv(os.path.join(ccx_sync_file_dir, file), dtype = str, skiprows = 1)

In [508]:
sync_contract.head(3)

,Synced,Organization,Manufacturer,Vendor,Contract #,ERP vendor #,ERP Manufacturer Name,ERP Manufacturer #,Contract Ref ID,Contract Desc,Source
0,False,Saint Lukes Cornwall Hospital (NEW),"105th Medical Squadron, Stewart Air National G...","105th Medical Squadron, Stewart Air National G...",1022658,NaN,NaN,NaN,46616391,Training Agreement,Local
1,False,Saint Lukes Cornwall Hospital (NEW),"105th Medical Squadron, Stewart Air National G...","105th Medical Squadron, Stewart Air National G...",1023416,NaN,NaN,NaN,49310881,Training Agreement for Air Force Medical Techn...,Local
2,False,Montefiore Information Technology,314e Corporation,314e Corporation,1000573,NaN,NaN,NaN,44100702,Entity Specific Local Contract;217.pdf;Main Co...,Local


In [509]:
sync_contract = sync_contract.fillna('')
sync_contract['report stamp'] = today

In [510]:
sync_contract['Synced'] = sync_contract['Synced'].apply(lambda x: 1 if x == 'True' else 0)
sync_contract['Organization'] = sync_contract['Organization'].apply(lambda x: x.strip())

In [511]:
# upload
table_to_load = sync_contract
table_name = '[Preprocessor].CCXSyncContract'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

In [512]:
# move loaded file to archive folder
for file in os.listdir(ccx_sync_file_dir):
    if ccx_sync in file:
        shutil.move(os.path.join(ccx_sync_file_dir, file),
                    os.path.join(ccx_sync_file_dir, 'archive', file))

# ContractLineError

In [513]:
cl_error = pd.read_csv(r'C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\temp export\ContractLineError.csv', dtype = str)

In [514]:
cl_error['ErrorMessageNumber'] = cl_error['ErrorMessageNumber'].astype(int)
cl_error['ContractLineError'] = cl_error['ContractLineError'].astype(int)
cl_error['ContractLine.UOM'] = cl_error['ContractLine.UOM'].apply(lambda x: x.strip())
cl_error['ContractLine.UOMConversion'] = cl_error['ContractLine.UOMConversion'].apply(lambda x: int(float(x.replace(',',''))))
cl_error['create stamp'] = pd.to_datetime(cl_error['create stamp'])
cl_error['update stamp'] = pd.to_datetime(cl_error['update stamp'])

In [515]:
cl_error = cl_error.fillna('')

In [516]:
# upload
table_to_load = cl_error
table_name = '[Preprocessor].inforcontractlineerror_stg'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# VendorLocation

In [517]:
vendor_loc = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\VendorLocation_Preprocessor.csv",
                         dtype = str)

In [518]:
vendor_loc['VendorLocationText'] = vendor_loc['RepresentativeText'].apply(lambda x: x.split(' - ')[-1])

In [519]:
col_to_include = ['Vendor', 'VendorName', 'VendorLocation', 
                 'VendorLocationText', 'Status', 'LocationType',
                 'create stamp', 'update stamp']

In [520]:
vendor_loc = vendor_loc[col_to_include].copy()

In [521]:
for col in ['create stamp', 'update stamp']:
    vendor_loc[col] = pd.to_datetime(vendor_loc[col])

vendor_loc = vendor_loc.fillna('')

In [522]:
# upload
table_to_load = vendor_loc
table_name = '[Preprocessor].InforVendorLocation'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# ItemReplenishFrom

In [523]:
item_replenish = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\ItemReplenishFrom.csv",
                            dtype = str)

In [524]:
item_replenish['ItemReplenishmentSource.ReplenishmentPriority'] = item_replenish['ItemReplenishmentSource.ReplenishmentPriority'].astype(int)
for col in ['create stamp', 'update stamp']:
    item_replenish[col] = pd.to_datetime(item_replenish[col])

item_replenish = item_replenish.fillna('')

In [525]:
# upload
table_to_load = item_replenish
table_name = '[Preprocessor].InforItemReplenishFrom'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 2 chunks will get inserted.
chunk [0, 49999] insertion completed.
chunk [50000, 99999] insertion completed.
Insertion completed.


0

# ItemGTIN

In [526]:
igtin = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\ItemGTIN.csv",
                   dtype = str)

In [527]:
igtin['UnitOfMeasure.UOMConversion'] = igtin['UnitOfMeasure.UOMConversion'].apply(lambda x: float(str(x).replace(',','')))

for col in ['create stamp', 'update stamp']:
    igtin[col] = pd.to_datetime(igtin[col])

igtin = igtin.fillna('')

In [528]:
# upload
table_to_load = igtin
table_name = '[DM_MONTYNT\\dli2].[MDM_ITEMGTIN]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Requesting Location

In [529]:
rloc = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\requesting+location.csv")

In [530]:
cols_to_take = ['Company', 'RequestingLocation', 'Active',
               'DerivedFinanceDimension1', 'DerivedFinanceDimension3', 'DerivedFinanceDimension5', 'DerivedProject',
               'FillOrKill', 'LocationRule', 'DerivedFromLocation', 'FromCompanyLocation',
               'PostalAddress', 'PostalAddress.DisplayAddressLine1', 'PostalAddress.DisplayAddressLine2', 'Name',
               'RequisitionApprovalType', 'create stamp', 'update stamp']

In [531]:
rloc = rloc[cols_to_take].copy()

In [532]:
for col in ['create stamp', 'update stamp']:
    rloc[col] = pd.to_datetime(rloc[col])

rloc = rloc.fillna('')

In [533]:
rloc.loc[:, 'ReportDate'] = datetime.now().strftime('%Y-%m-%d')

In [534]:
# upload
table_to_load = rloc
table_name = '[DM_MONTYNT\\dli2].[MDM_REQUESTING_LOCATION]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Commodity GL

In [347]:
cgl = pd.read_csv(r'C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\CommodityCodeGL.csv',
                  dtype = str)

In [348]:
cgl.drop(columns = ['ItemGroup'], inplace = True)

In [349]:
cgl.loc[:, 'ReportDate'] = datetime.now().strftime('%Y-%m-%d')
cgl = cgl.fillna('')
cgl.groupby(['CommodityCode']).size().sort_values()

CommodityCode
RX88-20    1
RX88-24    1
RX88-28    1
RX92       1
RX92-04    1
          ..
RX92-56    1
RX92-92    1
RX94       1
RX96       1
1000       1
Length: 8781, dtype: int64

In [350]:
# upload
table_to_load = cgl
table_name = '[DM_MONTYNT\\dli2].MDM_COMMODITY_GL'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Requester

In [351]:
requester = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\copy+of+requester.csv",
                         dtype = str)

In [352]:
requester.drop(columns = ['Reference1', 'Reference2'], inplace = True)

In [353]:
requester.loc[:, 'ReportDate'] = datetime.now().strftime('%Y-%m-%d')

In [354]:
requester.fillna('', inplace = True)

,Requester,Requester.Employee.PresentationNameSnapshot,RequestingLocation,OverrideContractCost,OverrideLastPOLastCost,ManualOverrideAllowed,EmailAddress,ReportDate
0,605,"Mariano, Anita",RMOS02060,No,Yes,Yes,amariano@montefiore.org,2026-06-02
1,976,"Walters, Delores",RWLR04988,No,Yes,Yes,dwalters@montefiore.org,2026-06-02
2,1015,"Laban-Grant, Olgica",RWPH07178,No,Yes,Yes,olaban@wphospital.org,2026-06-02
3,1239,"Pasa, Maria",RTCH07102,No,Yes,Yes,mpasa@montefiore.org,2026-06-02
4,1337,"Powell, Socorro",ROFF09872,No,Yes,Yes,spowell@montefiore.org,2026-06-02
...,...,...,...,...,...,...,...,...
3332,810480706,"NIOLA, VANESSA",ROFF10660,No,Yes,Yes,vanessa.niola@einsteinmed.edu,2026-06-02
3333,999967945,"Morales-Grajales, Neva",ROFF03300,No,Yes,Yes,nmorale1@montefiore.org,2026-06-02
3334,3100003314,"Cabrera, Michelle",RMHS07535,No,Yes,Yes,micabrer@montefiore.org,2026-06-02
3335,3100145175,"Robles, Katherine",ROFF06862,No,Yes,Yes,,2026-06-02


In [355]:
# upload
table_to_load = requester
table_name = '[DM_MONTYNT\\dli2].MDM_REQUESTER'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

# Complete Invoice

In [356]:
ci = pd.read_excel(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\Completed Invoices by Date.xlsx",
                  dtype = str, skiprows = 1)

cols_to_drop = ['Vendor Group', 'Document Group', 'Terms', 'Company Code2', 'PaymentText']
date_cols = ['Invoice Date', 'Due Date']
datetime_cols = ['Received Date', 'Date Created', 'Completed Date']

ci.drop(columns = cols_to_drop, inplace = True)

ci = type_conversion('date', ci, date_cols)
ci = type_conversion('datetime', ci, datetime_cols)
ci = type_conversion('float', ci, ['Invoice Amt'])

In [357]:
for col in ci.columns:
    if col in date_cols + datetime_cols + ['Invioce Amt']:
        continue
    else:
        ci[col] = ci[col].fillna('')
        ci[col] = ci[col].apply(lambda x: str(x).strip())

In [358]:
print(ci['Completed Date'].min(), ci['Completed Date'].max(), ci.shape)

2026-01-15 00:21:11 2026-04-17 08:06:08 (97125, 20)


In [359]:
ci.groupby(['Catalyst ID']).size().sort_values()

Catalyst ID
36287021    1
36287277    1
36287393    1
36287519    1
36287545    1
           ..
36288940    1
36288999    1
36289016    1
36289694    1
22489804    1
Length: 97125, dtype: int64

In [360]:
stupid_po_ind = ci[ci['PO Number'].apply(lambda x: len(x)) > 40].index
ci.loc[stupid_po_ind, 'PO Number'] = ''

In [350]:
# upload
table_to_load = ci
table_name = '[DM_MONTYNT\\dli2].ghx_completed_invoice_by_date_stg'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 2 chunks will get inserted.
chunk [0, 49999] insertion completed.
chunk [50000, 99999] insertion completed.
Insertion completed.


0

# GHX EXCLUDE GLD

In [351]:
geg = pd.read_excel(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\GHX_EXCLUDE_GLD_20250630.xlsx",
                   dtype = str)
geg.columns = ['Contract Number', 'Mfg Part Num', 'Supplier', 'GHX UOM']

In [352]:
table_to_load = geg
table_name = '[DM_MONTYNT\\dli2].PreprocessorGHX_EXCLUDE_GLD'
truncate_table(cnxn, table_name)

# === Connect to SQL Server ===
cursor = cnxn.cursor()

# === Insert rows one by one ===
insert_sql = f"""
    INSERT INTO {table_name} ([Contract Number], [Mfg Part Num], [Supplier], [GHX UOM])
    VALUES (?, ?, ?, ?)
"""

for index, row in geg.iterrows():
    cursor.execute(insert_sql, row['Contract Number'], row['Mfg Part Num'], row['Supplier'], row['GHX UOM'])

cnxn.commit()
cursor.close()
cnxn.close()
print("Insert complete.")

table truncated.
Insert complete.


# unload report

In [1222]:
ur = pd.read_csv(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\Unload-Report-09-06-2025.csv",
                  dtype = str, skiprows = 8)

ur.loc[:, 'Unload Date'] = pd.to_datetime(ur['Unload Date'], errors = 'coerce')
ur.loc[:, 'Tracking #'] = ur['Tracking #'].fillna('').astype(str)
ur.loc[:, 'PO Lines'] = ur['PO Lines'].fillna('').astype(str)
ur.loc[:, 'PO #'] = ur['PO #'].fillna('').astype(str)

TypeError: Invalid value '<DatetimeArray>
['2025-01-28 08:02:00', '2025-01-28 08:03:00', '2025-01-28 08:03:00',
 '2025-01-28 09:05:00', '2025-01-28 09:09:00', '2025-01-28 09:11:00',
 '2025-01-28 09:15:00', '2025-01-28 09:18:00', '2025-01-28 09:18:00',
 '2025-01-28 09:18:00',
 ...
 '2025-06-09 08:46:00', '2025-06-09 08:46:00', '2025-06-09 08:46:00',
 '2025-06-09 08:46:00', '2025-06-09 08:46:00', '2025-06-09 08:46:00',
 '2025-06-09 08:46:00', '2025-06-09 08:46:00', '2025-06-09 08:46:00',
 '2025-06-09 08:47:00']
Length: 56109, dtype: datetime64[us]' for dtype 'str'

In [ ]:
ur.loc[:, 'PO #'] = ur['PO #'].apply(lambda x: x.split('-')[0].strip()[:10] if x != '' else x)

In [ ]:
# upload
table_to_load = ur
table_name = '[DM_MONTYNT\\dli2].INNERTRACK_UNLOAD'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

# GL vocab

In [ ]:
gl_vocab = pd.read_excel(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\S13-APPO by vendor\Vendor Group\GL_vocab.xlsx",
                        dtype = str)

In [ ]:
gl_vocab.fillna('', inplace = True)
gl_vocab.loc[:, 'create stamp'] = '2024-02-12'
gl_vocab.loc[:, 'update stamp'] = '2025-02-18'

In [ ]:
gl_vocab_to_load = gl_vocab[['GL', 'GLText', 'CUSTOM', 'CLINICAL SUB CAT', 'create stamp', 'update stamp']].copy()

In [ ]:
table_to_load = gl_vocab_to_load
table_name = '[DM_MONTYNT\dli2].MDM_GL_SIMPLE_CLASSIFICATION'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

# Haoran's Medsurg

In [ ]:
ms_hj = pd.read_excel(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\S13-APPO by vendor\Vendor Group\All_Information_HJ.xlsx",
                     dtype = str,
                     sheet_name = 'MedSurg')

In [ ]:
ms_hj.loc[:, 'DISTRIBUTION_PCT'] = ms_hj['DISTRIBUTION_PCT'].astype(float)

In [ ]:
ms_hj.loc[:, 'create stamp'] = '2024-08-26'
ms_hj.loc[:, 'update stamp'] = '2024-08-26'

In [ ]:
ms_hj.groupby(['VENDOR', 'L0']).size().sort_values()

In [ ]:
ms_hj.drop_duplicates(subset = ['VENDOR', 'L0'], inplace = True)

In [ ]:
table_to_load = ms_hj
table_name = '[PRIME].[dbo].[MEDSURG_VENDOR_CLASSIFICATION_tbr]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

# Simi's CCX name mapping

In [ ]:
ccx_name = pd.read_excel(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\hardwork\CCX-Infor Contracted Vendor Mapping 1.xlsx",
                        dtype = str,
                        sheet_name = 'upload')

In [ ]:
ccx_name.drop(columns = ['VendorLocation_Infor'], inplace = True)
ccx_name = ccx_name.fillna('')
ccx_name = ccx_name.drop_duplicates()

In [ ]:
ccx_name.groupby(['ContractNumber_CCX', 'ManufacturerEID_CCX', 'Vendor_CCX', 'Vendor_Infor']).size().sort_values()

In [ ]:
ccx_name.loc[:, 'create date'] = '2024-08-28'
ccx_name.loc[:, 'update date'] = '2024-08-28'

In [ ]:
table_to_load = ccx_name
table_name = '[PRIME].[dbo].[CCX_INFOR_NAME_MATCHING_v1]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

# Korgun's PPE stuff

In [ ]:
PPE = pd.read_excel(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\P9-Backorder\PPE\PPE_MASTER_FILE_PREP_02052025.xlsx",
                   sheet_name = None, dtype = str)

In [ ]:
PPE.keys()

In [ ]:
date_col = ['LastUpdateDate']

PPE1 = PPE['PPE_ITEM']
for col in PPE1.columns:
    if col in date_col:
        PPE1.loc[:, col] = pd.to_datetime(PPE1[col], errors = 'coerce')
        PPE1.loc[:, col] = PPE1[col].apply(lambda x: x.strftime('%Y-%m-%d') if not pd.isnull(x) else '0001-01-01')
    else:
        PPE1.loc[:, col] = PPE1[col].fillna('')

PPE1.head(3)

In [ ]:
float_col = ['LTCMultiplier','AcuteMultiplier', 'Estimator']

PPE2 = PPE['PPE_CATEGORY']
for col in PPE2.columns:
    if col in float_col:
        PPE2.loc[:, col] = PPE2[col].astype(float)
        
PPE2.loc[:, 'LastUpdateDate'] = '2025-01-15'

PPE2.head(3)

In [ ]:
round(3.456, 2)

In [ ]:
int_col = ['InpatientBeds', 'StuffedBeds']
float_col = ['IMDCSTRMDist']

PPE3 = PPE['PPE_FACILITY']
for col in int_col:
    PPE3.loc[:, col] = PPE3[col].astype(int)
for col in float_col:
    PPE3.loc[:, col] = PPE3[col].astype(float)
PPE3.loc[:, 'Location'] = PPE3['Location'].fillna('')

PPE3.loc[:, 'LastUpdateDate'] = '2025-01-15'
PPE3.head(10)

In [ ]:
PPE4 = PPE['PPE_PARLOC']
PPE4.loc[:, 'LastUpdateDate'] = '2025-02-04'

PPE4.groupby(['Location']).size().sort_values()

In [ ]:
table_to_load = PPE1
table_name = '[DM_MONTYNT\dli2].[PPE_ITEM]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

In [ ]:
table_to_load = PPE2
table_name = '[DM_MONTYNT\dli2].[PPE_CATEGORY]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

In [ ]:
table_to_load = PPE3
table_name = '[DM_MONTYNT\dli2].[PPE_FACILITY]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

In [ ]:
table_to_load = PPE4
table_name = '[DM_MONTYNT\dli2].[PPE_PARLOC]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

# Preprocessor Company Map

In [469]:
prep_cmap = pd.read_excel(r"C:\Users\dli2\OneDrive - Montefiore Medicine\Documents\INFOR_SC\misc mdm\company_org_mapping.xlsx",
                          dtype = str)

In [471]:
prep_cmap['loaded stamp'] = datetime.today().strftime('%Y-%m-%d')

In [472]:
table_to_load = prep_cmap
table_name = '[Preprocessor].[CompanyOrganization]'
truncate_table(cnxn, table_name)
insert_to_prime_table(engine, cnxn, table_name, table_to_load, with_pkid = False, batch_size = 50000)

table truncated.
Total of 1 chunks will get inserted.
chunk [0, 49999] insertion completed.
Insertion completed.


0

In [172]:
import json

fp = r"I:\Procurement PMO\dli2\Monte PBO\beta\x.txt"
with open(fp, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

df.to_excel(r"I:\Procurement PMO\dli2\Monte PBO\beta\text.xlsx", index = False)